# 🤖 Autonomous Blog Generation Agent
## Built with LangGraph · FastAPI · Groq LLaMA · MongoDB · MLflow

This notebook implements the full **Autonomous Blog Generation Agent** pipeline:

| Phase | Description |
|-------|-------------|
| **Phase 0** | Environment setup & secret loading |
| **Phase 1** | State definition (`BlogState`, `Blog`) |
| **Phase 2** | LLM initialisation (Groq LLaMA‑3.1‑8b‑instant) |
| **Phase 3** | Node definitions (`BlogNode`) |
| **Phase 4** | Graph builder (`GraphBuilder`) |
| **Phase 5** | Run — topic‑only graph |
| **Phase 6** | Run — topic + language (Hindi & French) |
| **Phase 7** | MLflow experiment tracking |
| **Phase 8** | MongoDB persistence |
| **Phase 9** | Save all outputs → zip & download |

---

## 🔐 Colab Secrets required before running

Add the following keys in **Colab → 🔑 Secrets** (left sidebar):

| Secret name | Description |
|---|---|
| `GROQ_API_KEY` | Groq Cloud API key (already saved ✅) |
| `MONGO_DB_URL` | MongoDB Atlas connection string |
| `MLFLOW_TRACKING_URI` | DagsHub MLflow tracking URI |
| `MLFLOW_TRACKING_USERNAME` | DagsHub username |
| `MLFLOW_TRACKING_PASSWORD` | DagsHub access token / password |

> **Tip:** If you don't have DagsHub, set `USE_DAGSHUB = False` in Phase 0 and MLflow will log locally inside Colab.


## Phase 0 — Install dependencies & set environment variables

In [1]:
# ── Install all required packages ─────────────────────────
!pip install -q \
    langchain\
    langgraph \
    langchain-core \
    langchain-community \
    langchain-groq \
    pymongo \
    mlflow \
    dagshub \
    pydantic

print("✅ All packages installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 991.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/

In [2]:
import os
from google.colab import userdata

# ── Groq ──────────────────────────────────────────────────
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# ── MongoDB ───────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────
USE_DAGSHUB = True   # Set False to log MLflow locally

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")


✅ Env vars set.
   MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow


## Phase 1 — State Definitions (`BlogState` & `Blog`)

Mirrors `src/states/blogstate.py` exactly.


In [3]:
# ── src/states/blogstate.py ──────────────────────────────
from typing import TypedDict, Optional
from pydantic import BaseModel, Field


class Blog(BaseModel):
    """Structured output schema for a blog post."""
    title:   str = Field(description="The title of the blog post")
    content: str = Field(description="The main content of the blog post")


class BlogState(TypedDict):
    """LangGraph state shared across all nodes."""
    topic:            str
    blog:             dict           # serialisable Blog dict
    current_language: str


print("✅ Phase 1 — State definitions loaded.")
print("   BlogState keys : topic | blog | current_language")
print("   Blog fields    : title | content")


✅ Phase 1 — State definitions loaded.
   BlogState keys : topic | blog | current_language
   Blog fields    : title | content


## Phase 2 — LLM Initialisation

Mirrors `src/llms/groqllm.py`.  
Model: **`llama-3.1-8b-instant`** via Groq Cloud.


In [4]:
# ── src/llms/groqllm.py ──────────────────────────────────
from langchain_groq import ChatGroq


class GroqLLM:
    """Thin wrapper around LangChain's ChatGroq client."""

    def get_llm(self) -> ChatGroq:
        try:
            llm = ChatGroq(
                api_key=os.environ['GROQ_API_KEY'],
                model="llama-3.1-8b-instant",
            )
            return llm
        except Exception as e:
            raise ValueError(f"GroqLLM init error: {e}")


# ── Instantiate ───────────────────────────────────────────
groq_llm_obj = GroqLLM()
llm          = groq_llm_obj.get_llm()

print("✅ Phase 2 — LLM ready.")
print(f"   Model : {llm.model_name}")


✅ Phase 2 — LLM ready.
   Model : llama-3.1-8b-instant


## Phase 3 — Node Definitions (`BlogNode`)

Mirrors `src/nodes/blog_node.py`.

| Node method | Purpose |
|---|---|
| `title_creation` | Generate a creative, SEO-friendly blog title |
| `content_generation` | Generate detailed markdown blog content |
| `translation` | Translate content into a target language |
| `route` | Pass `current_language` downstream |
| `route_decision` | Conditional edge function → picks translation node |


In [9]:
import json
from langchain_core.messages import HumanMessage
from pydantic import ValidationError # Added for potential error handling


class BlogNode:
    """All LangGraph nodes for the blog generation pipeline."""

    def __init__(self, llm):
        self.llm = llm

    # ── Node 1 ──────────────────────────────────────────────
    def title_creation(self, state: BlogState) -> dict:
        """Generate a creative, SEO-friendly blog title."""
        if "topic" in state and state["topic"]:
            prompt = (
                "You are an expert blog content writer. Use Markdown formatting. "
                "Generate a blog title for the {topic}. "
                "This title should be creative and SEO friendly."
            ).format(topic=state["topic"])

            response = self.llm.invoke(prompt)
            print(f"   [title_creation] → {response.content[:80]}")
            return {"blog": {"title": response.content, "content": ""}}

    # ── Node 2 ──────────────────────────────────────────────
    def content_generation(self, state: BlogState) -> dict:
        """Generate detailed blog content in Markdown."""
        if "topic" in state and state["topic"]:
            system_prompt = (
                "You are an expert blog writer. Use Markdown formatting. "
                "Generate a detailed blog content with detailed breakdown "
                "for the {topic}."
            ).format(topic=state["topic"])

            response = self.llm.invoke(system_prompt)
            print(f"   [content_generation] → {len(response.content)} chars generated")
            return {
                "blog": {
                    "title":   state["blog"]["title"],
                    "content": response.content,
                }
            }

    # ── Node 3 ──────────────────────────────────────────────
    def translation(self, state: BlogState) -> dict:
        """Translate blog content into the specified language."""
        blog_title   = state["blog"]["title"]
        blog_content = state["blog"]["content"]
        lang         = state["current_language"]

        translation_prompt = (
            f"Translate the following blog post content and its title into {lang}.\n"
            "You must respond with a JSON object containing two keys: 'title' and 'content'.\n"
            "The 'title' key should contain the translated title.\n"
            "The 'content' key should contain the translated blog content.\n"
            "- Maintain the original tone, style, and formatting.\n"
            "- Adapt cultural references and idioms appropriately.\n\n"
            f"ORIGINAL TITLE: {blog_title}\n"
            f"ORIGINAL CONTENT:\n{blog_content}"
        )

        messages = [HumanMessage(content=translation_prompt)]

        response = self.llm.invoke(messages)
        # Attempt to parse the content as JSON
        try:
            # Some LLMs might wrap the JSON in markdown code blocks
            response_content_cleaned = response.content.strip()
            if response_content_cleaned.startswith("```json") and response_content_cleaned.endswith("```"):
                response_content_cleaned = response_content_cleaned[7:-3].strip()

            translated_blog_data = json.loads(response_content_cleaned)
            translated = Blog(
                title=translated_blog_data.get("title", blog_title), # Fallback to original title if not found
                content=translated_blog_data.get("content", "")
            )
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"   [translation:{lang}] Error parsing JSON: {e}. Raw response: {response.content[:200]}...")
            # Fallback: if JSON parsing fails, use original title and raw LLM response as content
            # This avoids crashing but indicates a translation formatting issue
            translated = Blog(title=f"[TRANSLATION ERROR] {blog_title}", content=f"Error translating content: {e}\n\nOriginal LLM Response: {response.content}")

        print(f"   [translation:{lang}] → {len(translated.content)} chars")
        return {
            "blog": {
                "title":   translated.title,
                "content": translated.content,
            }
        }

    # ── Router node ─────────────────────────────────────────
    def route(self, state: BlogState) -> dict:
        """Pass current_language downstream for conditional routing."""
        return {"current_language": state["current_language"]}

    # ── Conditional edge function ────────────────────────────
    def route_decision(self, state: BlogState) -> str:
        """Return the name of the next node based on language."""
        lang = state.get("current_language", "").lower()
        if lang == "hindi":
            return "hindi"
        elif lang == "french":
            return "french"
        else:
            return lang   # extensible for other languages

## Phase 4 — Graph Builder (`GraphBuilder`)

Mirrors `src/graphs/graph_builder.py`.

Two graph topologies:

```
Topic-only graph:
  START → title_creation → content_generation → END

Language graph:
  START → title_creation → content_generation → route ──┬──→ hindi_translation → END
                                                         └──→ french_translation → END
```


In [10]:
# ── src/graphs/graph_builder.py ──────────────────────────
from langgraph.graph import StateGraph, START, END


class GraphBuilder:
    """Builds and compiles LangGraph state-machine graphs."""

    def __init__(self, llm):
        self.llm  = llm
        self.blog_node_obj = BlogNode(self.llm)

    # ── Graph A: topic only ──────────────────────────────────
    def build_topic_graph(self) -> StateGraph:
        graph = StateGraph(BlogState)

        graph.add_node("title_creation",    self.blog_node_obj.title_creation)
        graph.add_node("content_generation", self.blog_node_obj.content_generation)

        graph.add_edge(START, "title_creation")
        graph.add_edge("title_creation", "content_generation")
        graph.add_edge("content_generation", END)

        return graph

    # ── Graph B: topic + language ────────────────────────────
    def build_language_graph(self) -> StateGraph:
        graph = StateGraph(BlogState)

        graph.add_node("title_creation",      self.blog_node_obj.title_creation)
        graph.add_node("content_generation",  self.blog_node_obj.content_generation)
        graph.add_node("route",               self.blog_node_obj.route)
        graph.add_node(
            "hindi_translation",
            lambda state: self.blog_node_obj.translation({**state, "current_language": "hindi"})
        )
        graph.add_node(
            "french_translation",
            lambda state: self.blog_node_obj.translation({**state, "current_language": "french"})
        )

        graph.add_edge(START, "title_creation")
        graph.add_edge("title_creation", "content_generation")
        graph.add_edge("content_generation", "route")

        graph.add_conditional_edges(
            "route",
            self.blog_node_obj.route_decision,
            {
                "hindi":  "hindi_translation",
                "french": "french_translation",
            }
        )
        graph.add_edge("hindi_translation",  END)
        graph.add_edge("french_translation", END)

        return graph

    # ── Compile helper ───────────────────────────────────────
    def setup_graph(self, usecase: str):
        if usecase == "topic":
            return self.build_topic_graph().compile()
        elif usecase == "language":
            return self.build_language_graph().compile()
        else:
            raise ValueError(f"Unknown usecase: {usecase}")


graph_builder = GraphBuilder(llm)
print("✅ Phase 4 — GraphBuilder defined.")


✅ Phase 4 — GraphBuilder defined.


## Phase 5 — Run: Topic-Only Graph

Generates a blog (title + content) for a topic in English.


In [11]:
import json, os
from IPython.display import display, Markdown

os.makedirs("outputs", exist_ok=True)

TOPIC = "Agentic AI"   # ✏️  change topic here

print(f"🚀 Running topic-only graph for topic: '{TOPIC}'")
topic_graph = graph_builder.setup_graph(usecase="topic")
state_topic = topic_graph.invoke({"topic": TOPIC})

title_topic   = state_topic["blog"]["title"]
content_topic = state_topic["blog"]["content"]

print("\n" + "="*60)
print("TITLE:", title_topic)
print("="*60)
print(content_topic[:500], "…")

# ── Save output ───────────────────────────────────────────
output_topic = {
    "usecase": "topic_only",
    "topic":   TOPIC,
    "blog":    {
        "title":   title_topic,
        "content": content_topic,
    }
}

with open("outputs/phase5_topic_output.json", "w") as f:
    json.dump(output_topic, f, indent=2)

with open("outputs/phase5_topic_blog.md", "w") as f:
    f.write(f"# {title_topic}\n\n{content_topic}")

print("\n✅ Phase 5 output saved → outputs/phase5_topic_output.json")
display(Markdown(f"**Title:** {title_topic}"))


🚀 Running topic-only graph for topic: 'Agentic AI'
   [title_creation] → **Unlocking the Future of AI: The Rise of Agentic AI**
   [content_generation] → 4464 chars generated

TITLE: **Unlocking the Future of AI: The Rise of Agentic AI**

**What is Agentic AI?**
---------------------

Agentic AI refers to a new wave of artificial intelligence that is designed to take a more proactive and autonomous approach to problem-solving. This type of AI is capable of making decisions and taking actions without being explicitly instructed, much like a human agent.

**Potential Applications of Agentic AI**
--------------------------------------

### 1. **Autonomous Systems**

Agentic AI can be used to create autonomous systems that can navigate complex environments, make decisions in real-time, and adapt to changing circumstances. This could lead to breakthroughs in areas such as robotics, self-driving cars, and smart homes.

### 2. **Personal Assistants**

Agentic AI can be used to create personal

**Title:** **Unlocking the Future of AI: The Rise of Agentic AI**
=====================================================

**What is Agentic AI?**
---------------------

Agentic AI refers to a new wave of artificial intelligence that is designed to take a more proactive and autonomous approach to problem-solving. This type of AI is capable of making decisions and taking actions without being explicitly instructed, much like a human agent.

**Potential Applications of Agentic AI**
--------------------------------------

### 1. **Autonomous Systems**

Agentic AI can be used to create autonomous systems that can navigate complex environments, make decisions in real-time, and adapt to changing circumstances. This could lead to breakthroughs in areas such as robotics, self-driving cars, and smart homes.

### 2. **Personal Assistants**

Agentic AI can be used to create personal assistants that can learn a user's habits and preferences, and make recommendations and decisions based on that information. This could lead to a new generation of personal assistants that are more proactive and helpful.

### 3. **Decision Support Systems**

Agentic AI can be used to create decision support systems that can analyze large amounts of data, identify patterns, and make recommendations based on that analysis. This could lead to breakthroughs in areas such as finance, healthcare, and marketing.

**Benefits of Agentic AI**
-------------------------

### 1. **Improved Efficiency**

Agentic AI can automate many routine tasks, freeing up human resources for more strategic and creative work.

### 2. **Enhanced Decision Making**

Agentic AI can analyze large amounts of data, identify patterns, and make recommendations based on that analysis, leading to more informed decision making.

### 3. **Increased Autonomy**

Agentic AI can take a more proactive and autonomous approach to problem-solving, leading to faster and more effective solutions.

**Challenges and Limitations of Agentic AI**
------------------------------------------

### 1. **Explainability**

Agentic AI can be difficult to explain and understand, making it challenging to ensure that decisions are fair and unbiased.

### 2. **Transparency**

Agentic AI can be opaque, making it challenging to understand how decisions are made and what data is being used.

### 3. **Security**

Agentic AI can be vulnerable to cyber attacks and data breaches, making it essential to ensure that systems are secure and protected.

**Conclusion**
----------

Agentic AI has the potential to revolutionize many areas of life, from autonomous systems to personal assistants and decision support systems. However, it also presents challenges and limitations that must be addressed. As we move forward with the development of agentic AI, it is essential to prioritize transparency, explainability, and security to ensure that these systems are fair, unbiased, and effective.

**Further Reading**
---------------

* [The Future of AI: A Brief History and a Look Ahead](https://www.example.com/future-of-ai)
* [The Ethics of AI: A Guide to the Challenges and Opportunities](https://www.example.com/ai-ethics)
* [Agentic AI: A New Era for Artificial Intelligence](https://www.example.com/agentic-ai)

## Phase 6 — Run: Language Graph (Hindi & French)

Generates a blog and then translates it into the chosen language.


In [12]:
# ── Hindi ─────────────────────────────────────────────────
TOPIC_LANG    = "NLP "   # ✏️ change topic
LANGUAGE_HINDI = "hindi"

print(f"🚀 Language graph: topic='{TOPIC_LANG}', language='{LANGUAGE_HINDI}'")
lang_graph = graph_builder.setup_graph(usecase="language")

state_hindi = lang_graph.invoke({
    "topic":            TOPIC_LANG,
    "current_language": LANGUAGE_HINDI,
})

title_hindi   = state_hindi["blog"]["title"]
content_hindi = state_hindi["blog"]["content"]

print("\nHindi Title :", title_hindi[:80])
print("Hindi Content snippet:", content_hindi[:200], "…")

output_hindi = {
    "usecase":  "language",
    "topic":    TOPIC_LANG,
    "language": LANGUAGE_HINDI,
    "blog":     {"title": title_hindi, "content": content_hindi}
}
with open("outputs/phase6_hindi_output.json", "w") as f:
    json.dump(output_hindi, f, indent=2, ensure_ascii=False)
with open("outputs/phase6_hindi_blog.md", "w", encoding="utf-8") as f:
    f.write(f"# {title_hindi}\n\n{content_hindi}")

print("✅ Hindi output saved → outputs/phase6_hindi_output.json")


🚀 Language graph: topic='NLP ', language='hindi'
   [title_creation] → **Unlocking the Power of Language: Emerging Trends in Natural Language Processin
   [content_generation] → 6122 chars generated
   [translation:hindi] Error parsing JSON: Expecting value: line 1 column 1 (char 0). Raw response: ```json
{
  "title": "नेचरल लैंग्वेज प्रोसेसिंग (NLP): तेजी से बढ़ती प्रगति और उनके अनुप्रयोग",
  "content": "
**टेबल ऑफ कंटेंट्स**

* [प्रारंभ](#प्रारंभ)
* [NLP क्या है?](#nlp-kyahai)
* [NLP का इतिहा...
   [translation:hindi] → 4191 chars

Hindi Title : [TRANSLATION ERROR] **Unlocking the Power of Language: Emerging Trends in Natura
Hindi Content snippet: Error translating content: Expecting value: line 1 column 1 (char 0)

Original LLM Response: ```json
{
  "title": "नेचरल लैंग्वेज प्रोसेसिंग (NLP): तेजी से बढ़ती प्रगति और उनके अनुप्रयोग",
  "content" …
✅ Hindi output saved → outputs/phase6_hindi_output.json


In [13]:
# ── French ────────────────────────────────────────────────
LANGUAGE_FRENCH = "french"

print(f"🚀 Language graph: topic='{TOPIC_LANG}', language='{LANGUAGE_FRENCH}'")
lang_graph_fr = graph_builder.setup_graph(usecase="language")

state_french = lang_graph_fr.invoke({
    "topic":            TOPIC_LANG,
    "current_language": LANGUAGE_FRENCH,
})

title_french   = state_french["blog"]["title"]
content_french = state_french["blog"]["content"]

print("\nFrench Title :", title_french[:80])
print("French Content snippet:", content_french[:200], "…")

output_french = {
    "usecase":  "language",
    "topic":    TOPIC_LANG,
    "language": LANGUAGE_FRENCH,
    "blog":     {"title": title_french, "content": content_french}
}
with open("outputs/phase6_french_output.json", "w") as f:
    json.dump(output_french, f, indent=2, ensure_ascii=False)
with open("outputs/phase6_french_blog.md", "w", encoding="utf-8") as f:
    f.write(f"# {title_french}\n\n{content_french}")

print("✅ French output saved → outputs/phase6_french_output.json")


🚀 Language graph: topic='NLP ', language='french'
   [title_creation] → # **"Revolutionizing Human-Machine Interaction: The Power of Natural Language Pr
   [content_generation] → 4716 chars generated
   [translation:french] Error parsing JSON: Expecting ',' delimiter: line 4 column 3 (char 203). Raw response: ```json
{
  "title": "# **\"Révolutionner l'Interaction Homme-Machine : Le Pouvoir du Traitement Automatique des Langues (TAL)\"**",
  "content": "**Traitement Automatique des Langues (TAL) : Guide Co...
   [translation:french] → 5889 chars

French Title : [TRANSLATION ERROR] # **"Revolutionizing Human-Machine Interaction: The Power of
French Content snippet: Error translating content: Expecting ',' delimiter: line 4 column 3 (char 203)

Original LLM Response: ```json
{
  "title": "# **\"Révolutionner l'Interaction Homme-Machine : Le Pouvoir du Traitement  …
✅ French output saved → outputs/phase6_french_output.json


## Phase 7 — MLflow Experiment Tracking

Logs each blog-generation run (topic, language, content length, title) as an MLflow experiment.


In [14]:
import mlflow, time

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment("autonomous_blog_generation")


def log_blog_run(run_name: str, params: dict, metrics: dict, artifacts: dict):
    """Log a single blog-generation run to MLflow."""
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for artifact_name, artifact_path in artifacts.items():
            mlflow.log_artifact(artifact_path, artifact_name)
    print(f"   ✅ MLflow run logged: {run_name}")


# ── Log topic-only run ────────────────────────────────────
log_blog_run(
    run_name="topic_only_english",
    params={
        "topic":    TOPIC,
        "usecase":  "topic_only",
        "language": "english",
        "model":    "llama-3.1-8b-instant",
    },
    metrics={
        "title_length":   len(title_topic),
        "content_length": len(content_topic),
        "timestamp":      time.time(),
    },
    artifacts={
        "blog_md":   "outputs/phase5_topic_blog.md",
        "blog_json": "outputs/phase5_topic_output.json",
    }
)

# ── Log Hindi run ─────────────────────────────────────────
log_blog_run(
    run_name="topic_language_hindi",
    params={
        "topic":    TOPIC_LANG,
        "usecase":  "language",
        "language": "hindi",
        "model":    "llama-3.1-8b-instant",
    },
    metrics={
        "title_length":   len(title_hindi),
        "content_length": len(content_hindi),
        "timestamp":      time.time(),
    },
    artifacts={
        "blog_md":   "outputs/phase6_hindi_blog.md",
        "blog_json": "outputs/phase6_hindi_output.json",
    }
)

# ── Log French run ────────────────────────────────────────
log_blog_run(
    run_name="topic_language_french",
    params={
        "topic":    TOPIC_LANG,
        "usecase":  "language",
        "language": "french",
        "model":    "llama-3.1-8b-instant",
    },
    metrics={
        "title_length":   len(title_french),
        "content_length": len(content_french),
        "timestamp":      time.time(),
    },
    artifacts={
        "blog_md":   "outputs/phase6_french_blog.md",
        "blog_json": "outputs/phase6_french_output.json",
    }
)

print("\n✅ Phase 7 — All MLflow runs logged.")


2026/04/13 03:52:29 INFO mlflow.tracking.fluent: Experiment with name 'autonomous_blog_generation' does not exist. Creating a new experiment.


🏃 View run topic_only_english at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/11/runs/41169811eb3146cfad7201623234b4f7
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/11
   ✅ MLflow run logged: topic_only_english
🏃 View run topic_language_hindi at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/11/runs/bcbfb618bb6747b988b1ea53fc9afd65
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/11
   ✅ MLflow run logged: topic_language_hindi
🏃 View run topic_language_french at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/11/runs/6fbb4021146246798ccfe52a6fa4a289
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/11
   ✅ MLflow run logged: topic_language_french

✅ Phase 7 — All MLflow runs logged.


## Phase 8 — MongoDB Persistence

Saves all generated blogs into a MongoDB collection `blogs` in database `blog_agent_db`.


In [15]:
from pymongo import MongoClient
from datetime import datetime

MONGO_URL = os.environ['MONGO_DB_URL']
client    = MongoClient(MONGO_URL)
db        = client["blog_agent_db"]
collection = db["blogs"]

def save_blog(record: dict) -> str:
    """Insert a blog record and return its inserted_id."""
    record["created_at"] = datetime.utcnow().isoformat()
    result = collection.insert_one(record)
    return str(result.inserted_id)


# ── Save topic-only English blog ──────────────────────────
id_en = save_blog({
    "usecase":  "topic_only",
    "topic":    TOPIC,
    "language": "english",
    "title":    title_topic,
    "content":  content_topic,
})
print(f"✅ English blog saved  → _id: {id_en}")

# ── Save Hindi blog ───────────────────────────────────────
id_hi = save_blog({
    "usecase":  "language",
    "topic":    TOPIC_LANG,
    "language": "hindi",
    "title":    title_hindi,
    "content":  content_hindi,
})
print(f"✅ Hindi blog saved    → _id: {id_hi}")

# ── Save French blog ──────────────────────────────────────
id_fr = save_blog({
    "usecase":  "language",
    "topic":    TOPIC_LANG,
    "language": "french",
    "title":    title_french,
    "content":  content_french,
})
print(f"✅ French blog saved   → _id: {id_fr}")

# ── Verify count ─────────────────────────────────────────
count = collection.count_documents({})
print(f"\n📊 Total docs in collection 'blogs': {count}")

client.close()
print("✅ Phase 8 — MongoDB persistence complete.")


/tmp/ipykernel_3675/3304684222.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record["created_at"] = datetime.utcnow().isoformat()


✅ English blog saved  → _id: 69dc68925893dc9e82bd738f
✅ Hindi blog saved    → _id: 69dc68945893dc9e82bd7390
✅ French blog saved   → _id: 69dc68945893dc9e82bd7391

📊 Total docs in collection 'blogs': 3
✅ Phase 8 — MongoDB persistence complete.


## Phase 9 — Save All Outputs, Zip & Download

Writes a final summary report, zips the entire `outputs/` folder, and triggers a browser download.


In [16]:
import zipfile, glob
from google.colab import files

# ── Final summary JSON ────────────────────────────────────
summary = {
    "project":      "Autonomous Blog Generation Agent",
    "model":        "llama-3.1-8b-instant (Groq)",
    "framework":    "LangGraph + LangChain >= 1.2",
    "runs": [
        {
            "usecase":  "topic_only",
            "topic":    TOPIC,
            "language": "english",
            "title":    title_topic,
            "mongo_id": id_en,
        },
        {
            "usecase":  "language",
            "topic":    TOPIC_LANG,
            "language": "hindi",
            "title":    title_hindi,
            "mongo_id": id_hi,
        },
        {
            "usecase":  "language",
            "topic":    TOPIC_LANG,
            "language": "french",
            "title":    title_french,
            "mongo_id": id_fr,
        },
    ]
}

with open("outputs/summary_all_phases.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("✅ summary_all_phases.json written.")

# ── Zip all outputs ───────────────────────────────────────
ZIP_PATH = "blog_agent_outputs.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(glob.glob("outputs/**", recursive=True)):
        if os.path.isfile(fpath):
            zf.write(fpath)
            print(f"   added: {fpath}")

print(f"\n📦 Zip created: {ZIP_PATH}")

# ── Download ──────────────────────────────────────────────
files.download(ZIP_PATH)
print("✅ Phase 9 — Download triggered.")


✅ summary_all_phases.json written.
   added: outputs/phase5_topic_blog.md
   added: outputs/phase5_topic_output.json
   added: outputs/phase6_french_blog.md
   added: outputs/phase6_french_output.json
   added: outputs/phase6_hindi_blog.md
   added: outputs/phase6_hindi_output.json
   added: outputs/summary_all_phases.json

📦 Zip created: blog_agent_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Phase 9 — Download triggered.
